# Terra Data Extraction Pipeline

This notebook implements the "Option 2" methodology: automating the extraction of Market Data (UST Supply, LUNA Mcap, LUNA Price) via CoinGecko API and preparing the structure for merging with Anchor data.

## Requirements
```bash
pip install pycoingecko pandas numpy matplotlib
```

In [ ]:
import pandas as pd
import numpy as np
from pycoingecko import CoinGeckoAPI
import time

cg = CoinGeckoAPI()

### 1. Market Data Fetcher (CoinGecko)

In [ ]:
def get_coingecko_history(coin_id, vs_currency='usd', days='max'):
    """
    Fetches historical market chart data from CoinGecko.
    Returns a DataFrame with daily closes for Market Cap and Price.
    """
    print(f"Fetching history for {coin_id}...")
    try:
        data = cg.get_coin_market_chart_by_id(id=coin_id, vs_currency=vs_currency, days=days)
    except Exception as e:
        print(f"Error fetching {coin_id}: {e}")
        return pd.DataFrame()
        
    # 1. Market Caps
    df_mc = pd.DataFrame(data['market_caps'], columns=['timestamp', 'market_cap'])
    df_mc['Date'] = pd.to_datetime(df_mc['timestamp'], unit='ms').dt.date
    # Aggregating by last value per day (Close)
    daily_mc = df_mc.groupby('Date')['market_cap'].last()

    # 2. Prices
    df_price = pd.DataFrame(data['prices'], columns=['timestamp', 'price'])
    df_price['Date'] = pd.to_datetime(df_price['timestamp'], unit='ms').dt.date
    daily_price = df_price.groupby('Date')['price'].last()

    # 3. Total Volumes
    df_vol = pd.DataFrame(data['total_volumes'], columns=['timestamp', 'volume'])
    df_vol['Date'] = pd.to_datetime(df_vol['timestamp'], unit='ms').dt.date
    daily_vol = df_vol.groupby('Date')['volume'].last()

    # Merge
    df_final = pd.DataFrame({
        f'{coin_id}_mcap': daily_mc,
        f'{coin_id}_price': daily_price,
        f'{coin_id}_volume': daily_vol
    }).reset_index()
    
    return df_final

### 2. Execution: Fetch UST and LUNA

In [ ]:
# Fetch Luna (Classic) and UST (Classic)
# Note: CoinGecko IDs are 'terra-luna' (Classic) and 'terrausd' (Classic)
# Check if IDs have changed; historically 'terra-luna' is typically LUNC now.
luna_data = get_coingecko_history('terra-luna')
time.sleep(1) # Rate limit politeness
ust_data = get_coingecko_history('terrausd')

# Rename columns for schema compatibility
luna_data = luna_data.rename(columns={
    'terra-luna_mcap': 'LUNA_MarketCap',
    'terra-luna_price': 'LUNA_Price',
    'terra-luna_volume': 'LUNA_Volume'
})

ust_data = ust_data.rename(columns={
    'terrausd_mcap': 'UST_MarketCap',
    'terrausd_price': 'UST_Price',
    'terrausd_volume': 'UST_Volume'
})
# UST Supply is roughly Market Cap / Price, or just Market Cap if Price ~ 1
# CoinGecko Market Cap is usually Circulating Supply * Price.
# So UST_Supply = UST_MarketCap / UST_Price (Safety check needed)
ust_data['UST_Supply'] = ust_data['UST_MarketCap'] / ust_data['UST_Price'].replace(0, 1)

# Merge
market_df = pd.merge(luna_data, ust_data, on='Date', how='outer').sort_values('Date')
print(f"Fetched {len(market_df)} daily records.")
market_df.tail()

### 3. Merging with Anchor Data (Instructions)

The Anchor Protocol data (Deposits/Borrows) must be fetched from Dune Analytics using the SQL query provided in `anchor_dune_queries.sql`.

1. Run the SQL on Dune.
2. Export result to `anchor_data.csv`.
3. Place `anchor_data.csv` in this directory.

The code below merges it.

In [ ]:
try:
    anchor_df = pd.read_csv('anchor_data.csv', parse_dates=['Date'])
    # Ensure Date column is just date part if needed
    anchor_df['Date'] = anchor_df['Date'].dt.date
    
    final_df = pd.merge(market_df, anchor_df, on='Date', how='left')
    print("Merged Anchor data successfully.")
except FileNotFoundError:
    print("WARNING: 'anchor_data.csv' not found. Dataframe will lack Anchor metrics.")
    final_df = market_df.copy()
    final_df['Anchor_Deposits'] = 0
    final_df['Anchor_Borrows'] = 0

### 4. Export to `terra_daily_metrics.csv`

In [ ]:
# Filter Date Range (Jan 2021 - May 2022)
mask = (final_df['Date'] >= pd.to_datetime('2021-01-01').date()) & (final_df['Date'] <= pd.to_datetime('2022-05-30').date())
final_df = final_df.loc[mask]

# Save
final_df.to_csv('terra_daily_metrics.csv', index=False)
print("Saved to terra_daily_metrics.csv")